# Очистка данных: Contacts

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

import help_130625_dam as h

# Автоматическая настройка путей: если мы в папке notebooks, выходим в корень проекта
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
    print(f"Рабочая директория изменена на корень: {os.getcwd()}")
else:
    print(f"Текущая рабочая директория: {os.getcwd()}")

# Пути к данным (теперь они всегда от корня проекта)
RAW_DATA_DIR    = 'Sources'
CLEANED_DIR     = os.path.join('data', 'cleaned')

CONTACTS_INPUT  = os.path.join(RAW_DATA_DIR, 'Contacts (Done).xlsx')
CONTACTS_OUTPUT = os.path.join(CLEANED_DIR, 'contacts_clean.pkl')

MAPPING_OUTPUT  = os.path.join(CLEANED_DIR, 'contacts_mapping.pkl')
BUYERS_INFO     = os.path.join(CLEANED_DIR, 'buyers_info.pkl')

DEALS_CLEAN     = os.path.join(CLEANED_DIR, 'deals_clean.pkl')
CALLS_CLEAN     = os.path.join(CLEANED_DIR, 'calls_clean.pkl')

pd.set_option('display.float_format', '{:.2f}'.format)
np.set_printoptions(suppress=True, precision=2)

## Загрузка и первичный осмотр

In [128]:
df = pd.read_excel(CONTACTS_INPUT, dtype={'Id': str})

# Переименование столбцов в snake_case
df.columns = [h.to_snake(c) for c in df.columns]

print(f'Форма: {df.shape}')

df['id'] = pd.array([h.str_to_int64(v) for v in df['id']], dtype='Int64')

n_before = df.shape[0]

h.descr_df(df, include='all', show_stats=False, show_sample_rows=False)

Форма: (18548, 4)


,Тип,Заполнено,Пропуски,% Пропусков,Уникальных
Признак,,,,,
id,Int64,18548,0,0.0,18548
contact_owner_name,object,18548,0,0.0,28
created_time,str,18548,0,0.0,17921
modified_time,str,18548,0,0.0,16580


### Загружаем информацию по покупателям (которую мы сохранили в 03_cleaning_deals)

In [129]:
if os.path.exists(BUYERS_INFO):
    buyers_info = pd.read_pickle(BUYERS_INFO)
    print(f"Загружены данные о {len(buyers_info)} клиентах из {BUYERS_INFO}")
    print(f"Колонки: {buyers_info.columns.tolist()}")
else:
    print(f"Файл {BUYERS_INFO} не найден! Проверьте, отработала ли ячейка экспорта в 03_cleaning_deals.")

Загружены данные о 17007 клиентах из ../data/cleaned/buyers_info.pkl
Колонки: ['id', 'is_buyer', 'first_payment_date', 'first_any_deal_date', 'new_registration_date']


In [139]:
print(f"Число бизнес-дубликатов: {df.duplicated(subset=df.columns[1:]).sum()}")

Число бизнес-дубликатов: 22


После нескольких итераций и разбора источника пришел к выводу:
Пропусков нет. Дубликатов 38. Важно ИД зафиксирован как Int64. Есть один менеджер "False" у которого у которого один клиент. В сделках у этого клиента другой менеджер. Значит этот менеджер техническая ошибка. Заменяем менеджера на Jane Smith, так как она первая в звонках у этого клиента.

In [130]:
# Исправляем менеджера "False". 
mask_false = df['contact_owner_name'].astype(str).str.lower() == 'false'
if mask_false.any():
    target_id = df.loc[mask_false, 'id'].iloc[0]
    df.loc[mask_false, 'contact_owner_name'] = 'Jane Smith'
    print(f"Менеджер исправлен для контакта {target_id}")

# Обработка дублей с перепривязкой сделок и звонков
# Находим дубликаты по всем полям кроме ID
cols_to_check = [c for c in df.columns if c != 'id']
is_dupe = df.duplicated(subset=cols_to_check, keep=False)

if is_dupe.any():
    # Оставляем последнюю запись как "мастер-карточку"
    df_masters = df[is_dupe].sort_values('id').drop_duplicates(subset=cols_to_check, keep='last')
    
    mapping = {}
    for _, master_row in df_masters.iterrows():
        mask = True
        for col in cols_to_check:
            if pd.isna(master_row[col]):
                mask &= df[col].isna()
            else:
                mask &= (df[col] == master_row[col])
        
        all_ids = df.loc[mask, 'id'].tolist()
        m_id = master_row['id']
        
        for oid in all_ids:
            if oid != m_id:
                mapping[oid] = m_id

    if mapping:
        print(f"Сформирован маппинг для {len(mapping)} дублей.")
        
        # Сохраняем маппинг на диск — будет использован в 03 и 04
        os.makedirs(os.path.dirname(MAPPING_OUTPUT), exist_ok=True)
        pd.Series(mapping).to_pickle(MAPPING_OUTPUT)
        print(f"Маппинг сохранён: {MAPPING_OUTPUT}")
        
        # Перепривязываем уже существующие файлы (если есть)
        if os.path.exists(DEALS_CLEAN):
            deals_tmp = pd.read_pickle(DEALS_CLEAN)
            affected = deals_tmp['contact_id'].isin(mapping.keys()).sum()
            deals_tmp['contact_id'] = deals_tmp['contact_id'].replace(mapping)
            deals_tmp.to_pickle(DEALS_CLEAN)
            print(f"Deals: Дополнительно перепривязано {affected} сделок к мастер-контактам.")

        if os.path.exists(CALLS_CLEAN):
            calls_tmp = pd.read_pickle(CALLS_CLEAN)
            # В звонках id контакта может называться contactid
            c_col = 'contactid' if 'contactid' in calls_tmp.columns else 'contact_id'
            if c_col in calls_tmp.columns:
                calls_tmp[c_col] = pd.to_numeric(calls_tmp[c_col], errors='coerce').astype('Int64')
                affected_calls = calls_tmp[c_col].isin(mapping.keys()).sum()
                calls_tmp[c_col] = calls_tmp[c_col].replace(mapping)
                calls_tmp.to_pickle(CALLS_CLEAN)
                print(f"Calls: Дополнительно перепривязано {affected_calls} звонков.")

    # Удаляем дубликаты из основного DataFrame
    df = df.drop_duplicates(subset=cols_to_check, keep='last').reset_index(drop=True)
    print(f"В таблице Contacts {len(df)} уникальных клиентов.")
else:
    mapping = {}
    print("Бизнес-дубликатов не обнаружено.")

if 'target_id' in locals() and target_id in mapping:
    print(f"Контакт {target_id} успешно перепривязан к мастеру {mapping[target_id]}.")

Менеджер исправлен для контакта 5805028000008772190
Сформирован маппинг для 38 дублей.
Маппинг сохранён: ../data/cleaned/contacts_mapping.pkl
В таблице Contacts 18510 уникальных клиентов.


## Типы данных: даты

In [143]:
DATE_COLS = ['created_time', 'modified_time']

for col in DATE_COLS:
    # Пробуем автоопределение формата (dayfirst=True для DD.MM.YYYY)
    df[col] = pd.to_datetime(df[col], dayfirst=True, errors='coerce')

# Проверяем результат
print('Типы после парсинга:')
print(df[DATE_COLS].dtypes)
print()

# Считаем NaT = не распарсились
for col in DATE_COLS:
    nat_count = df[col].isna().sum()
    print(f'{col}: NaT = {nat_count} ({nat_count/len(df)*100:.2f}%)')

Типы после парсинга:
created_time     datetime64[us]
modified_time    datetime64[us]
dtype: object

created_time: NaT = 0 (0.00%)
modified_time: NaT = 0 (0.00%)


In [132]:
# Преобразование в категориальный тип для оптимизации
df['contact_owner_name'] = df['contact_owner_name'].astype('category')

owner_counts = df['contact_owner_name'].value_counts()
print(f'Уникальных менеджеров: {owner_counts.shape[0]}')

Уникальных менеджеров: 27


### Обогащение данных

In [133]:
# Обогащаем основную таблицу контактов данными из сделок
# Перед слиянием удаляем уже существующие (старые/ошибочные) колонки, если они есть в df
cols_to_add = ['is_buyer', 'first_payment_date', 'first_any_deal_date', 'new_registration_date']
df = df.drop(columns=[c for c in cols_to_add if c in df.columns])

# Мы используем id, который теперь совпадает в обеих таблицах
df = df.merge(buyers_info[['id'] + cols_to_add], on='id', how='left')

# Заполняем пропуски в is_buyer (кто не попал в buyers_info - не покупатель)
df['is_buyer'] = df['is_buyer'].fillna(False).astype(bool)

# Корректируем дату регистрации там, где она была зафиксирована раньше создания контакта
# (если new_registration_date не пустое, используем его)
mask = df['new_registration_date'].notna()
affected = mask.sum()
df.loc[mask, 'created_time'] = df.loc[mask, 'new_registration_date']

# Удаляем временные колонки после коррекции
df = df.drop(columns=['new_registration_date', 'first_any_deal_date'])

print(f"Контакты обогащены данными: добавлена дата первой оплаты для {df['first_payment_date'].notna().sum()} клиентов.")
print(f"Дата регистрации скорректирована для {affected} контактов.")
print(f"is_buyer: {df['is_buyer'].sum()} покупателей из {len(df)} контактов ({df['is_buyer'].mean():.1%})")

Контакты обогащены данными: добавлена дата первой оплаты для 816 клиентов.
Дата регистрации скорректирована для 2911 контактов.
is_buyer: 816 покупателей из 18510 контактов (4.4%)


## Итоговый осмотр

In [135]:
h.descr_df(df, include='all', show_stats=True, show_sample_rows=True, show_quartiles=True)

,Тип,Заполнено,Пропуски,% Пропусков,Уникальных,Пример 1,Пример 2,Пример 3,Min,Mean,Median,Max,Range,Q1,Q3,IQR
Признак,,,,,,,,,,,,,,,,
id,Int64,18510,0,0.00,18510,5805028000000645014,5805028000000872003,5805028000000889001,5805028000000645014,5805028000029634560.0,5805028000029577216.0,5805028000056907001,56261632.0,5805028000016978944.0,5805028000043587584.0,26608640.0
contact_owner_name,category,18510,0,0.00,27,Rachel White,Charlie Davis,Bob Brown,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,<NA>,<NA>
created_time,datetime64[us],18510,0,0.00,17819,2023-06-27 11:28:00,2023-07-03 11:31:00,2023-07-02 22:37:00,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,<NA>,<NA>
modified_time,datetime64[us],18510,0,0.00,16580,2023-12-22 13:34:00,2024-05-21 10:23:00,2023-12-21 13:17:00,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,<NA>,<NA>
is_buyer,bool,18510,0,0.00,2,False,False,False,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,<NA>,<NA>
first_payment_date,datetime64[us],816,17694,95.59,808,NaT,NaT,NaT,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,<NA>,<NA>


## Сохранение

In [142]:
os.makedirs(os.path.dirname(CONTACTS_OUTPUT), exist_ok=True)
df.to_pickle(CONTACTS_OUTPUT)
df.to_excel(CONTACTS_OUTPUT.replace('.pkl', '.xlsx'))
# Итоги
summary_data = {
    'Метрика': [
        'Строк исходно',
        'Строк после очистки',
        'Удалено дубликатов',
        'Диапазон дат',
        'Уникальных менеджеров',
        'Уникальных клиентов',
    ],
    'Значение': [
        n_before,
        len(df),
        n_before - len(df),
        f'{df["created_time"].min().date()} → {df["created_time"].max().date()}',
        df['contact_owner_name'].nunique(),
        f'{df["is_buyer"].sum():,.0f}',
    ]
}

print(f'Сохранено: {CONTACTS_OUTPUT}')
display(pd.DataFrame(summary_data))

Сохранено: ../data/cleaned/contacts_clean.pkl


,Метрика,Значение
0,Строк исходно,18548
1,Строк после очистки,18510
2,Удалено дубликатов,38
3,Диапазон дат,2023-06-27 → 2024-06-21
4,Уникальных менеджеров,27
5,Уникальных клиентов,816


## Описание датасета

**Источник:** `Contacts (Done).xlsx` — выгрузка из CRM  
**Назначение:** справочник лидов/клиентов; связывает сделки и звонки с конкретным контактом через `id`

| Столбец | Тип | Описание |
|---|---|---|
| `id` | `Int64` | Уникальный ID контакта в CRM (19-значный) |
| `contact_owner_name` | `category` | Менеджер, ответственный за контакт |
| `created_time` | `datetime` | Дата и время регистрации лида |
| `modified_time` | `datetime` | Дата последнего изменения записи |
| `first_payment_date` | `datetime` | **Обогащённый признак:** дата самого раннего платежа (из `buyers_info.pkl`); NaN = лид без оплаты |
| `is_buyer` | `bool` | **Обогащённый признак:** (из `buyers_info.pkl`); True если контакт совершил хотя бы одну оплату  (`first_payment_date` не пустая) |
| `new_registration_date` | `datetime` | **Обогащённый признак:** (из `buyers_info.pkl`); Дата сделки в случае когда первая сделка раньше регистрации контакта |

**Объём:** ~18 510 контактов, 7 столбцов после обогащения  
**Ключевые связи:**
- `id` → `deals.contact_id` (сделки)
- `id` → `calls.contactid` (звонки)

## Выводы

В исходных данных CRM обнаружено **38 дублирующихся контактов** и технически некорректный менеджер `"False"` (1 запись). Дубли появляются при повторном создании контакта в CRM — например, когда клиент обращается повторно и менеджер регистрирует новую карточку вместо поиска существующей. Менеджер `"False"` — результат программного сбоя при импорте данных.

Без очистки каждый такой контакт учитывался бы как отдельный лид: метрика конверсии (Leads → Buyers) была бы искусственно занижена, а связанные сделки и звонки не объединялись бы на одного клиента — корректный расчёт LTV и анализ воронки были бы невозможны.

**Что сделано:** дубли удалены с перепривязкой сделок и звонков к мастер-ID через `contacts_mapping.pkl`; менеджер исправлен с `"False"` на `Jane Smith` (первый контакт по звонкам). Датасет обогащён флагом `is_buyer` и датой первого платежа `first_payment_date` из `buyers_info.pkl`.

> **Системная рекомендация:** настроить в CRM валидацию на дублирование контактов по номеру телефона или email ещё на этапе ввода данных. Менеджеров обязать при обнаружении дубля сразу же принимать меры к объединению контактов.